# Webserv - Commandes de tests

Toutes les commandes `curl` pour tester le serveur HTTP.

## Prérequis

```bash
# 1. Compiler le serveur
make re

# 2. Lancer le serveur
./webserv config/default.conf

# 3. Dans un autre terminal, lancer les tests
```

---
## Script automatise (35 tests)

Le script compile, lance le serveur, execute les tests, puis arrete le serveur automatiquement.

```bash
# Rendre executable (une seule fois)
chmod +x tests/run_tests.sh

# Lancer tous les tests
./tests/run_tests.sh
```

### Couverture du script

| Section | Tests | Description |
|---------|-------|-------------|
| 1. GET statiques | 1-4 | index, fichier direct, 404, page erreur |
| 2. Redirections | 5 | 301 + verification header Location |
| 3. POST upload | 6-9 | upload, lecture contenu, ecrasement |
| 4. DELETE | 10-12 | suppression, verification, fichier inexistant |
| 5. Autoindex | 13-14 | ON (HTML verifie), OFF (403) |
| 6. CGI | 15-18 | Python GET, query string, Bash, POST |
| 7. Methodes 405 | 19-20 | DELETE interdit, POST sans script |
| 8. Headers HTTP | 21-24 | Content-Type, keep-alive, close |
| 9. Body size 413 | 25-26 | body > limit = 413, petit body OK |
| 10. Non-bloquant | 27 | CGI + statique en parallele |
| 11. Second serveur | 28-29 | port 8081 GET, POST interdit |
| 12. Edge cases | 30-33 | URI longue, double slash, stabilite |
| 13. Pages erreur | 34-35 | 404 et 405 custom |

Le script retourne le nombre d'echecs comme code de sortie (`exit $FAIL`).

---

## 1. GET - Fichiers statiques

### 1.1 Page d'accueil
```bash
curl -v http://localhost:8080/
```
**Attendu** : `200 OK` + contenu de `www/index.html`

### 1.2 Fichier spécifique
```bash
curl -v http://localhost:8080/index.html
```
**Attendu** : `200 OK`

### 1.3 Fichier inexistant
```bash
curl -v http://localhost:8080/cette-page-nexiste-pas
```
**Attendu** : `404 Not Found`

### 1.4 Page d'erreur personnalisée
```bash
curl -v http://localhost:8080/errors/404.html
```
**Attendu** : `200 OK` (le fichier d'erreur existe en tant que fichier statique)

## 2. GET - Redirections

### 2.1 Redirection 301
```bash
curl -v http://localhost:8080/redirect
```
**Attendu** : `301 Moved Permanently` + header `Location: https://www.google.com`

### 2.2 Suivre la redirection
```bash
curl -v -L http://localhost:8080/redirect
```
**Attendu** : Suit le redirect vers Google (`-L` = follow redirects)

### 2.3 Voir uniquement les headers
```bash
curl -I http://localhost:8080/redirect
```
**Attendu** : Headers uniquement, dont `Location:`

## 3. POST - Upload de fichiers

### 3.1 Upload simple (texte)
```bash
curl -v -X POST http://localhost:8080/uploads/mon_fichier.txt \
  -H "Content-Type: text/plain" \
  -d "Contenu de mon fichier de test"
```
**Attendu** : `201 Created` → fichier créé dans `www/uploads/mon_fichier.txt`

### 3.2 Vérifier le fichier uploadé
```bash
curl -v http://localhost:8080/uploads/mon_fichier.txt
```
**Attendu** : `200 OK` + `Contenu de mon fichier de test`

### 3.3 Upload avec Content-Disposition
```bash
curl -v -X POST http://localhost:8080/uploads/ \
  -H "Content-Type: text/plain" \
  -H 'Content-Disposition: attachment; filename="upload_header.txt"' \
  -d "Upload via header Content-Disposition"
```
**Attendu** : `201 Created` → fichier `www/uploads/upload_header.txt`

### 3.4 Upload binaire (image)
```bash
curl -v -X POST http://localhost:8080/uploads/image.png \
  -H "Content-Type: image/png" \
  --data-binary @/chemin/vers/image.png
```
**Attendu** : `201 Created`

## 4. DELETE - Suppression de fichiers

### 4.1 Supprimer un fichier existant
```bash
curl -v -X DELETE http://localhost:8080/uploads/mon_fichier.txt
```
**Attendu** : `204 No Content`

### 4.2 Verifier la suppression
```bash
curl -v http://localhost:8080/uploads/mon_fichier.txt
```
**Attendu** : `404 Not Found`

### 4.3 Supprimer un fichier inexistant
```bash
curl -v -X DELETE http://localhost:8080/uploads/nexiste_pas.txt
```
**Attendu** : `404 Not Found`

### 4.4 DELETE non autorise sur /
```bash
curl -v -X DELETE http://localhost:8080/index.html
```
**Attendu** : `405 Method Not Allowed` (DELETE pas dans `allowed_methods` de `/`)

## 5. Autoindex (listing de répertoire)

### 5.1 Autoindex activé (/uploads/)
```bash
curl -v http://localhost:8080/uploads/
```
**Attendu** : `200 OK` + page HTML avec tableau des fichiers du répertoire

### 5.2 Autoindex désactivé (/cgi-test/)
```bash
curl -v http://localhost:8080/cgi-test/
```
**Attendu** : `403 Forbidden`

### 5.3 Voir le rendu dans un navigateur
```
Ouvrir : http://localhost:8080/uploads/
```
**Attendu** : Tableau HTML avec colonnes Nom, Taille, Date de modification

## 6. CGI

### 6.1 CGI Python - GET simple
```bash
curl -v http://localhost:8080/cgi-test/test.py
```
**Attendu** : `200 OK` + HTML généré par Python avec variables d'environnement CGI

### 6.2 CGI Python - GET avec query string
```bash
curl -v "http://localhost:8080/cgi-test/test.py?name=webserv&lang=cpp"
```
**Attendu** : `200 OK` + section "Paramètres GET" avec `name=webserv`, `lang=cpp`

### 6.3 CGI Python - POST
```bash
curl -v -X POST http://localhost:8080/cgi-test/test.py \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "user=test&action=login"
```
**Attendu** : `200 OK` + section "Body POST reçu" avec `user=test&action=login`

### 6.4 CGI Bash - GET
```bash
curl -v http://localhost:8080/cgi-test/test.sh
```
**Attendu** : `200 OK` + HTML avec date, user, hostname

### 6.5 CGI - Script inexistant
```bash
curl -v http://localhost:8080/cgi-test/inexistant.py
```
**Attendu** : `404 Not Found`

## 7. Méthodes non autorisées

### 7.1 DELETE sur / (non autorisé)
```bash
curl -v -X DELETE http://localhost:8080/index.html
```
**Attendu** : `405 Method Not Allowed`

## 8. Headers et comportement HTTP

### 8.1 Voir les headers de reponse
```bash
curl -i http://localhost:8080/
```
**Attendu** : `Content-Type: text/html`, `Content-Length`

### 8.2 Keep-Alive
```bash
curl -v -H "Connection: keep-alive" http://localhost:8080/
```
**Attendu** : `200 OK`, connexion maintenue

### 8.3 Connection: close
```bash
curl -v -H "Connection: close" http://localhost:8080/
```
**Attendu** : `200 OK`, connexion fermee apres la reponse

### 8.4 Content-Type selon l'extension
```bash
curl -si http://localhost:8080/index.html | grep Content-Type
# → Content-Type: text/html
```

---

## 8bis. client_max_body_size (413)

### Config
- Port 8080 : `client_max_body_size 10M` (serveur), `50M` (location /uploads)
- Port 8081 : `client_max_body_size 5M` (serveur), `1M` (location /api)

### 8bis.1 Body trop gros (> 5M sur port 8081)
```bash
# Generer un fichier de 6MB
python3 -c "import sys; sys.stdout.buffer.write(b'X' * (1024 * 1024 * 6))" > /tmp/bigbody.bin

curl -v -X POST http://localhost:8081/ \
  -H "Content-Type: application/octet-stream" \
  --data-binary @/tmp/bigbody.bin

rm -f /tmp/bigbody.bin
```
**Attendu** : `413 Payload Too Large`

### 8bis.2 Body normal (ne doit PAS renvoyer 413)
```bash
curl -v -X POST http://localhost:8080/cgi-test/test.py \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "data=ok"
```
**Attendu** : `200 OK`

## 9. CGI non-bloquant (test de concurrence)

### 9.1 Deux requêtes en parallèle
```bash
# Lancer un CGI et une requête statique en même temps
curl -s -w "CGI: %{time_total}s\n" http://localhost:8080/cgi-test/test.py &
curl -s -w "Static: %{time_total}s\n" http://localhost:8080/ &
wait
```
**Attendu** : La requête statique termine **avant** le CGI → prouve le non-bloquant

### 9.2 Plusieurs CGI en parallèle
```bash
for i in 1 2 3 4 5; do
  curl -s -w "CGI $i: %{time_total}s\n" -o /dev/null \
    "http://localhost:8080/cgi-test/test.py?id=$i" &
done
wait
```
**Attendu** : Les 5 CGI s'exécutent en parallèle (temps similaires, pas séquentiels)

### 9.3 Requêtes statiques pendant un CGI lent
```bash
# Terminal 1 : CGI lent
curl -v http://localhost:8080/cgi-test/test.py

# Terminal 2 (en même temps) : requêtes rapides
curl -w "Time: %{time_total}s\n" http://localhost:8080/
curl -w "Time: %{time_total}s\n" http://localhost:8080/index.html
```
**Attendu** : Les requêtes statiques répondent instantanément même si le CGI tourne

## 10. Tests avec un navigateur

Ouvrir dans Firefox/Chrome :

| URL | Attendu |
|-----|--------|
| `http://localhost:8080/` | Page d'accueil |
| `http://localhost:8080/uploads/` | Listing des fichiers (autoindex) |
| `http://localhost:8080/cgi-test/test.py` | Page CGI Python |
| `http://localhost:8080/cgi-test/test.py?name=42&lang=cpp` | CGI avec paramètres |
| `http://localhost:8080/cgi-test/test.sh` | Page CGI Bash |
| `http://localhost:8080/redirect` | Redirection vers Google |
| `http://localhost:8080/nexiste-pas` | Page d'erreur 404 |

## 11. Edge cases et requêtes malformées

### 11.1 URI très longue
```bash
curl -v http://localhost:8080/$(python3 -c "print('a'*5000)")
```
**Attendu** : `414 URI Too Long` ou `404`

### 11.2 Requête avec body vide en POST
```bash
curl -v -X POST http://localhost:8080/uploads/empty.txt \
  -H "Content-Type: text/plain" \
  -d ""
```
**Attendu** : `201 Created` (fichier vide créé)

### 11.3 Double slash dans URI
```bash
curl -v http://localhost:8080//index.html
```
**Attendu** : `200 OK` (normalisé) ou `404`

### 11.4 Path traversal (sécurité)
```bash
curl -v http://localhost:8080/../../../etc/passwd
```
**Attendu** : `403 Forbidden` ou `400 Bad Request` (ne doit JAMAIS renvoyer le fichier)

### 11.5 Requête HTTP brute via telnet/nc
```bash
echo -e "GET / HTTP/1.1\r\nHost: localhost\r\n\r\n" | nc localhost 8080
```
**Attendu** : Réponse HTTP complète avec headers et body

### 11.6 Requête incomplète (timeout)
```bash
# Envoyer une requête sans la terminer
echo -ne "GET / HTTP/1.1\r\nHost: localhost\r\n" | nc localhost 8080
# Attendre... le serveur doit timeout après quelques secondes
```
**Attendu** : Le serveur ferme la connexion après le timeout (60s par défaut)

## 12. Tests de charge (stress)

### 12.1 Avec siege
```bash
# Installer siege
sudo apt install siege

# 50 utilisateurs concurrent pendant 10 secondes
siege -c 50 -t 10s http://localhost:8080/
```
**Attendu** : Availability > 99%, pas de crash du serveur

### 12.2 Avec ab (Apache Bench)
```bash
# 1000 requêtes, 10 concurrentes
ab -n 1000 -c 10 http://localhost:8080/
```
**Attendu** : 0 erreurs, temps de réponse < 100ms

### 12.3 Boucle rapide en bash
```bash
for i in $(seq 1 100); do
  curl -s -o /dev/null -w "%{http_code} " http://localhost:8080/
done
echo ""
```
**Attendu** : 100x `200` sans aucun échec

## 12bis. Surveillance RAM (fuites memoire)

### Verifier que le serveur ne bouffe pas la RAM

```bash
# One-shot : voir la RAM du serveur
ps -o pid,vsz,rss,pmem,comm -p $(pgrep -f "./webserv")
```

| Colonne | Signification |
|---------|---------------|
| `VSZ` | Memoire virtuelle (KB) |
| `RSS` | **Memoire physique reelle (KB)** — c'est celle qui compte |
| `%MEM` | Pourcentage de la RAM totale |

### Surveillance en temps reel (toutes les 2 secondes)

```bash
watch -n 2 'ps -o pid,vsz,rss,pmem,comm -p $(pgrep -f "./webserv")'
```

### Procedure complete : avant/apres siege

```bash
# Terminal 1 : surveiller la RAM en continu
watch -n 1 'ps -o pid,rss,pmem -p $(pgrep -f "./webserv")'

# Terminal 2 : stresser le serveur
siege -c 100 -r 50 -b http://localhost:8080/
```

**Attendu** : Le RSS reste stable autour de ~3-4 MB avant et apres siege. Si ca monte sans redescendre, il y a une fuite memoire.

## 13. Port 8081 (second serveur)

### 13.1 GET sur port 8081
```bash
curl -v http://localhost:8081/
```
**Attendu** : `200 OK` (serveur `example.com`)

### 13.2 POST non autorisé sur port 8081
```bash
curl -v -X POST http://localhost:8081/ -d "data"
```
**Attendu** : `405 Method Not Allowed` (seul GET autorisé sur `/` du port 8081)

### 13.3 Autoindex sur port 8081
```bash
curl -v http://localhost:8081/
```
**Attendu** : `200 OK` avec autoindex (si pas d'index.html) car `autoindex on`

## 14. Cookies & Sessions

### Principe

Le serveur gere des sessions cote serveur avec un cookie `session_id`.
- Login : cree une session + envoie un cookie `Set-Cookie: session_id=sess_XXX`
- Dashboard : page protegee, verifie le cookie → 401 si invalide
- Logout : detruit la session + expire le cookie
- Nettoyage automatique des sessions expirees toutes les 60s dans la boucle `poll()`

### Flux complet

```
POST /login (username=bob)
  → Set-Cookie: session_id=sess_XXX; Max-Age=3600; Path=/; HttpOnly; SameSite=Lax
  → 302 Redirect → /dashboard

GET /dashboard (Cookie: session_id=sess_XXX)
  → Verifie la session → 200 + page "Bienvenue, bob"

GET /logout (Cookie: session_id=sess_XXX)
  → Supprime la session → Set-Cookie: session_id=; Max-Age=0
  → 302 Redirect → /login.html

GET /dashboard (sans cookie ou session expiree)
  → 401 Unauthorized
```

### 14.1 Login - creer une session
```bash
curl -v -X POST -d "username=testuser" http://localhost:8080/login
```
**Attendu** : `302 Found` + header `Set-Cookie: session_id=sess_XXXX...` + `Location: /dashboard`

### 14.2 Dashboard avec cookie valide
```bash
# Remplacer sess_XXXX par le session_id recu au login
curl -v -b "session_id=sess_XXXX" http://localhost:8080/dashboard
```
**Attendu** : `200 OK` + page HTML avec "Bienvenue, testuser"

### 14.3 Dashboard SANS cookie
```bash
curl -v http://localhost:8080/dashboard
```
**Attendu** : `401 Unauthorized`

### 14.4 Dashboard avec FAUX cookie
```bash
curl -v -b "session_id=fake123456" http://localhost:8080/dashboard
```
**Attendu** : `401 Unauthorized`

### 14.5 Logout
```bash
curl -v -b "session_id=sess_XXXX" http://localhost:8080/logout
```
**Attendu** : `302 Found` + `Set-Cookie: session_id=; Max-Age=0` + `Location: /login.html`

### 14.6 Dashboard apres logout (session detruite)
```bash
curl -v -b "session_id=sess_XXXX" http://localhost:8080/dashboard
```
**Attendu** : `401 Unauthorized`

### 14.7 Test expiration automatique des sessions

Pour tester rapidement, reduire temporairement les valeurs dans `Server.cpp` :

```cpp
// Server.cpp - dans la boucle run()
// Changer :
//   now - last_cleanup > 60       →  now - last_cleanup > 10
//   cleanupExpiredSessions(3600)  →  cleanupExpiredSessions(20)
```

Puis :

```bash
# 1. Recompiler et lancer le serveur
make re && ./webserv config/default.conf

# 2. Login
curl -v -X POST -d "username=expire_test" http://localhost:8080/login

# 3. Verifier que ca marche
curl -v -b "session_id=sess_XXXX" http://localhost:8080/dashboard
# → 200 OK

# 4. Attendre 30 secondes, observer les logs serveur :
#    [SERVER] Nettoyage des sessions

# 5. Re-tenter avec le meme cookie
curl -v -b "session_id=sess_XXXX" http://localhost:8080/dashboard
# → 401 Unauthorized (session expiree et nettoyee)
```

### 14.8 Test complet dans le navigateur

| Etape | Action | Resultat attendu |
|-------|--------|-----------------|
| 1 | Ouvrir `http://localhost:8080/` | Page accueil avec carte "Cookies & Sessions" |
| 2 | Cliquer sur `/login.html` | Formulaire de connexion |
| 3 | Entrer un nom + Se connecter | Redirect vers dashboard avec message "Bienvenue" |
| 4 | Cliquer "Se deconnecter" | Redirect vers login.html |
| 5 | Acceder directement a `http://localhost:8080/dashboard` | 401 Unauthorized |

### Resultats attendus

| Test | Code HTTP | Description |
|------|-----------|-------------|
| POST /login avec username | 302 | Cookie `session_id` cree |
| GET /dashboard avec cookie valide | 200 | Page personnalisee |
| GET /dashboard sans cookie | 401 | Acces refuse |
| GET /dashboard avec faux cookie | 401 | Acces refuse |
| GET /logout avec cookie | 302 | Session detruite, cookie expire |
| GET /dashboard apres logout | 401 | Session n'existe plus |
| GET /dashboard apres expiration | 401 | Session nettoyee automatiquement |

## Resume rapide - Copier/coller

```bash
# ---- Tout tester rapidement ----

# Statique
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/             # 200
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/nope          # 404

# Redirect
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/redirect      # 301

# Upload + verif + delete
curl -s -o /dev/null -w "%{http_code}" -X POST http://localhost:8080/uploads/t.txt -H "Content-Type: text/plain" -d "test"  # 201
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/uploads/t.txt                                                   # 200
curl -s -o /dev/null -w "%{http_code}" -X DELETE http://localhost:8080/uploads/t.txt                                         # 204
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/uploads/t.txt                                                   # 404

# Autoindex
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/uploads/      # 200
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/cgi-test/     # 403

# CGI
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/cgi-test/test.py   # 200
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/cgi-test/test.sh   # 200
curl -s -o /dev/null -w "%{http_code}" -X POST http://localhost:8080/cgi-test/test.py -d "data=ok"  # 200

# Methode non autorisee
curl -s -o /dev/null -w "%{http_code}" -X DELETE http://localhost:8080/index.html  # 405

# Cookies & Sessions
curl -s -o /dev/null -w "%{http_code}" -X POST -d "username=test" http://localhost:8080/login          # 302
curl -s -o /dev/null -w "%{http_code}" http://localhost:8080/dashboard                                  # 401
curl -s -o /dev/null -w "%{http_code}" -b "session_id=fake" http://localhost:8080/dashboard             # 401

# Second serveur
curl -s -o /dev/null -w "%{http_code}" http://localhost:8081/                        # 200
curl -s -o /dev/null -w "%{http_code}" -X POST http://localhost:8081/ -d "data"      # 405

# Ou plus simple : lancer le script automatise
./tests/run_tests.sh   # 35 tests, compile + lance le serveur tout seul
```